In [1]:
!nvidia-smi

Wed Sep 30 08:36:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes pandas tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.3 MB/s eta 0:00:00


In [3]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [4]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/snli")

print(dataset)

README.md:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  412kB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  413kB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/550152 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 550152
    })
})


In [5]:
test = dataset["test"]

print("Total test examples:", len(test))
print(test[0])

Total test examples: 10000
{'premise': 'This church choir sings to the masses as they sing joyous songs from the book at a church.', 'hypothesis': 'The church has cracks in the ceiling.', 'label': 1}


In [6]:
# Remove examples with invalid labels
test_clean = test.filter(lambda x: x["label"] != -1)

print("Valid test examples:", len(test_clean))

# Take the first 500 examples
N = 500
test_subset = test_clean.select(range(N))

print("Our experiment size:", len(test_subset))

# Show the first example
print(test_subset[0])

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Valid test examples: 9824
Our experiment size: 500
{'premise': 'This church choir sings to the masses as they sing joyous songs from the book at a church.', 'hypothesis': 'The church has cracks in the ceiling.', 'label': 1}


In [7]:
label_names = {
    0: "ENTAILMENT",
    1: "NEUTRAL",
    2: "CONTRADICTION"
}

print(label_names)

{0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}


In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


model_name = "cross-encoder/nli-deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name)

model = model.to("cuda")
model.eval()

print("Model loaded!")
print(model.config.id2label)

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Model loaded!
{0: 'contradiction', 1: 'entailment', 2: 'neutral'}


In [9]:
# Take the first SNLI example
example = test_subset[0]

premise = example["premise"]
hypothesis = example["hypothesis"]

print("Premise:", premise)
print("Hypothesis:", hypothesis)
print("True label:", label_names[example["label"]])

Premise: This church choir sings to the masses as they sing joyous songs from the book at a church.
Hypothesis: The church has cracks in the ceiling.
True label: NEUTRAL


In [10]:
# Tokenize the premise and hypothesis
inputs = tokenizer(
    premise,
    hypothesis,
    return_tensors="pt",
    truncation=True
)

# Move inputs to GPU
inputs = {key: value.to("cuda") for key, value in inputs.items()}

# Run DeBERTa
with torch.no_grad():
    outputs = model(**inputs)

# Get the predicted class
prediction_id = torch.argmax(outputs.logits, dim=-1).item()

# Convert number to label
prediction = model.config.id2label[prediction_id]

print("Predicted label:", prediction)

Predicted label: neutral


In [11]:
def predict_deberta(premise, hypothesis):
    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True
    )

    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    prediction_id = torch.argmax(
        outputs.logits, dim=-1
    ).item()

    return model.config.id2label[prediction_id].upper()

In [12]:
from tqdm.auto import tqdm

deberta_predictions = []

for example in tqdm(test_subset):

    prediction = predict_deberta(
        example["premise"],
        example["hypothesis"]
    )

    deberta_predictions.append(prediction)

  0%|          | 0/500 [00:00<?, ?it/s]

In [13]:
correct = 0

for example, prediction in zip(
    test_subset,
    deberta_predictions
):
    true_label = label_names[example["label"]]

    if prediction == true_label:
        correct += 1

accuracy = correct / len(test_subset)

print("Correct:", correct)
print("Total:", len(test_subset))
print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

Correct: 460
Total: 500
Accuracy: 0.92
Accuracy (%): 92.0


In [15]:
deberta_accuracy = accuracy

print("DeBERTa accuracy:", deberta_accuracy * 100, "%")

DeBERTa accuracy: 92.0 %


In [16]:
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM
import torch

mistral_name = "mistralai/Mistral-7B-Instruct-v0.2"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

mistral_tokenizer = AutoTokenizer.from_pretrained(mistral_name)

mistral = AutoModelForCausalLM.from_pretrained(
    mistral_name,
    quantization_config=quant_config,
    device_map="auto"
)

mistral.eval()

print("Mistral loaded!")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Mistral loaded!


In [49]:
import torch

SYSTEM_PROMPT = """You are doing natural language inference.
Given a premise and a hypothesis, decide whether the hypothesis is:

- Yes: definitely true given the premise (the premise guarantees it).
- No: definitely false given the premise (the premise rules it out).
- Maybe: could be true or false; the premise does not settle it.

Examples:
Premise: A man is playing a guitar on stage.
Hypothesis: A person is making music.
Answer: Yes

Premise: A man is playing a guitar on stage.
Hypothesis: The man is asleep in bed.
Answer: No

Premise: A man is playing a guitar on stage.
Hypothesis: The man is famous.
Answer: Maybe

Premise: Two kids are swimming in a pool.
Hypothesis: Children are in the water.
Answer: Yes

Premise: Two kids are swimming in a pool.
Hypothesis: The kids are playing soccer.
Answer: No

Premise: Two kids are swimming in a pool.
Hypothesis: The kids are siblings.
Answer: Maybe

Reply with exactly one word: Yes, No, or Maybe."""

answer_to_label = {"Yes": "ENTAILMENT", "No": "CONTRADICTION", "Maybe": "NEUTRAL"}

answer_token_ids = {
    w: mistral_tokenizer.encode(w, add_special_tokens=False)[-1]
    for w in answer_to_label
}
print("Token ids:", answer_token_ids)   # should be 3 different numbers


def predict_mistral(premise, hypothesis):
    user_msg = f"{SYSTEM_PROMPT}\n\nPremise: {premise}\nHypothesis: {hypothesis}\nAnswer:"
    formatted = mistral_tokenizer.apply_chat_template(
        [{"role": "user", "content": user_msg}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = mistral_tokenizer(formatted, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(mistral.device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = mistral(**inputs).logits[0, -1]

    scores = {w: logits[i].item() for w, i in answer_token_ids.items()}
    best = max(scores, key=scores.get)
    return answer_to_label[best], scores

Token ids: {'Yes': 5592, 'No': 1770, 'Maybe': 5833}


In [50]:
sanity_examples = [
    {"premise": "A dog is running.", "hypothesis": "An animal is running.", "expected": "ENTAILMENT"},
    {"premise": "A dog is running.", "hypothesis": "No animal is moving.", "expected": "CONTRADICTION"},
    {"premise": "A dog is running.", "hypothesis": "The dog is wearing a red hat.", "expected": "NEUTRAL"},
]

for ex in sanity_examples:
    pred, scores = predict_mistral(ex["premise"], ex["hypothesis"])
    print("Expected:", ex["expected"], "| Mistral:", pred)
    print("  scores:", {k: round(v, 2) for k, v in scores.items()})

Expected: ENTAILMENT | Mistral: ENTAILMENT
  scores: {'Yes': 25.0, 'No': 4.09, 'Maybe': 8.38}
Expected: CONTRADICTION | Mistral: CONTRADICTION
  scores: {'Yes': 6.25, 'No': 22.0, 'Maybe': 10.69}
Expected: NEUTRAL | Mistral: NEUTRAL
  scores: {'Yes': 5.31, 'No': 16.25, 'Maybe': 24.25}


In [51]:
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

mistral_preds = []
for ex in tqdm(test_subset):
    pred, _ = predict_mistral(ex["premise"], ex["hypothesis"])
    mistral_preds.append(pred)

true_labels = [label_names[ex["label"]] for ex in test_subset]

mistral_accuracy = sum(p == t for p, t in zip(mistral_preds, true_labels)) / len(true_labels)
print("Mistral accuracy:", round(mistral_accuracy * 100, 2), "%")
print()
print(classification_report(true_labels, mistral_preds))

order = ["ENTAILMENT", "CONTRADICTION", "NEUTRAL"]
print("Confusion matrix (rows = true, cols = predicted):")
print(order)
print(confusion_matrix(true_labels, mistral_preds, labels=order))

  0%|          | 0/500 [00:00<?, ?it/s]

Mistral accuracy: 76.8 %

               precision    recall  f1-score   support

CONTRADICTION       0.94      0.70      0.80       164
   ENTAILMENT       0.74      0.98      0.85       174
      NEUTRAL       0.66      0.61      0.64       162

     accuracy                           0.77       500
    macro avg       0.78      0.76      0.76       500
 weighted avg       0.78      0.77      0.76       500

Confusion matrix (rows = true, cols = predicted):
['ENTAILMENT', 'CONTRADICTION', 'NEUTRAL']
[[171   0   3]
 [  3 114  47]
 [ 56   7  99]]


In [52]:
print(f"DeBERTa: {deberta_accuracy * 100:.2f}%")
print(f"Mistral: {mistral_accuracy * 100:.2f}%")

DeBERTa: 92.00%
Mistral: 76.80%


In [53]:
!pip install -q nvidia-ml-py

In [54]:
import pynvml, time, numpy as np, torch
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

def measure_run(predict_fn, examples):
    """One full pass over the examples. Returns accuracy, latency, energy."""
    torch.cuda.synchronize()
    e0 = pynvml.nvmlDeviceGetTotalEnergyConsumption(handle)   # millijoules since driver load
    t0 = time.perf_counter()

    preds, lats = [], []
    for ex in examples:
        s = time.perf_counter()
        preds.append(predict_fn(ex["premise"], ex["hypothesis"]))
        torch.cuda.synchronize()
        lats.append(time.perf_counter() - s)

    total_time = time.perf_counter() - t0
    e1 = pynvml.nvmlDeviceGetTotalEnergyConsumption(handle)

    energy_j = (e1 - e0) / 1000.0
    n = len(examples)
    true = [label_names[ex["label"]] for ex in examples]
    acc = sum(p == t for p, t in zip(preds, true)) / n
    return {
        "accuracy": acc,
        "runtime_s": total_time,
        "avg_latency_ms": 1000 * float(np.mean(lats)),
        "energy_total_J": energy_j,
        "energy_per_query_J": energy_j / n,
        "avg_gpu_power_W": energy_j / total_time,
        "peak_mem_GB": torch.cuda.max_memory_allocated() / 1024**3,
    }

# wrappers so both models return just the label string
deberta_fn = lambda p, h: predict_deberta(p, h)
mistral_fn = lambda p, h: predict_mistral(p, h)[0]

In [56]:
# Warm-up: first CUDA calls include kernel compilation and allocator setup
for ex in test_subset.select(range(20)):
    deberta_fn(ex["premise"], ex["hypothesis"])
    mistral_fn(ex["premise"], ex["hypothesis"])

import pandas as pd
results = []
for run in range(5):
    for name, fn in [("DeBERTa-v3-base", deberta_fn), ("Mistral-7B (few-shot)", mistral_fn)]:
        r = measure_run(fn, test_subset)
        r["model"] = name; r["run"] = run
        results.append(r)
        print(run, name, round(r["accuracy"], 3), round(r["energy_per_query_J"], 3), "J/query")

df = pd.DataFrame(results)

0 DeBERTa-v3-base 0.92 2.262 J/query
0 Mistral-7B (few-shot) 0.768 39.527 J/query
1 DeBERTa-v3-base 0.92 1.387 J/query
1 Mistral-7B (few-shot) 0.768 39.29 J/query
2 DeBERTa-v3-base 0.92 1.445 J/query
2 Mistral-7B (few-shot) 0.768 39.305 J/query
3 DeBERTa-v3-base 0.92 1.413 J/query
3 Mistral-7B (few-shot) 0.768 39.329 J/query
4 DeBERTa-v3-base 0.92 1.393 J/query
4 Mistral-7B (few-shot) 0.768 39.291 J/query


In [57]:
# Keep the 500-sample results in case you want them later
df.to_csv("nli_energy_runs_500.csv", index=False)

N = 1000
test_subset = test_clean.shuffle(seed=42).select(range(N))

# sanity checks
print("Subset size:", len(test_subset))
from collections import Counter
print(Counter(label_names[l] for l in test_subset["label"]))   # should be roughly balanced

Subset size: 1000
Counter({'NEUTRAL': 336, 'ENTAILMENT': 334, 'CONTRADICTION': 330})


In [58]:
# Short warm-up (model is already loaded; this just settles GPU clocks)
for ex in test_subset.select(range(20)):
    deberta_fn(ex["premise"], ex["hypothesis"])
    mistral_fn(ex["premise"], ex["hypothesis"])

results = []
for run in range(5):
    for name, fn in [("DeBERTa-v3-base", deberta_fn), ("Mistral-7B (few-shot)", mistral_fn)]:
        r = measure_run(fn, test_subset)
        r["model"] = name; r["run"] = run
        results.append(r)
        print(run, name, round(r["accuracy"], 3), round(r["energy_per_query_J"], 3), "J/query")

df = pd.DataFrame(results)
df.to_csv("nli_energy_runs_1000.csv", index=False)

0 DeBERTa-v3-base 0.924 1.391 J/query
0 Mistral-7B (few-shot) 0.768 39.219 J/query
1 DeBERTa-v3-base 0.924 1.374 J/query
1 Mistral-7B (few-shot) 0.768 39.241 J/query
2 DeBERTa-v3-base 0.924 1.368 J/query
2 Mistral-7B (few-shot) 0.768 39.309 J/query
3 DeBERTa-v3-base 0.924 1.363 J/query
3 Mistral-7B (few-shot) 0.768 39.296 J/query
4 DeBERTa-v3-base 0.924 1.373 J/query
4 Mistral-7B (few-shot) 0.768 39.313 J/query


In [59]:
import numpy as np
GRID_KG_PER_KWH = 0.7   # assumed; look up and cite a source for your region

df["gCO2e_per_query"] = df["energy_per_query_J"] / 3.6e6 * GRID_KG_PER_KWH * 1000

summary = df.groupby("model").agg(
    accuracy=("accuracy", "mean"),
    latency_ms=("avg_latency_ms", "mean"),
    latency_std=("avg_latency_ms", "std"),
    J_per_query=("energy_per_query_J", "mean"),
    J_std=("energy_per_query_J", "std"),
    gCO2e_per_query=("gCO2e_per_query", "mean"),
    avg_power_W=("avg_gpu_power_W", "mean"),
    peak_mem_GB=("peak_mem_GB", "max"),
).round(5)

# 95% CI for accuracy (normal approximation, n = 1000)
summary["acc_ci95"] = (1.96 * np.sqrt(summary["accuracy"] * (1 - summary["accuracy"]) / N)).round(4)

print(summary.T)
ratio = summary.loc["Mistral-7B (few-shot)", "J_per_query"] / summary.loc["DeBERTa-v3-base", "J_per_query"]
print(f"\nMistral uses {ratio:.1f}x more energy per query than DeBERTa")
summary.to_csv("nli_summary_1000.csv")

model            DeBERTa-v3-base  Mistral-7B (few-shot)
accuracy                 0.92400                0.76800
latency_ms              24.85314              572.15004
latency_std              0.29844                0.75295
J_per_query              1.37371               39.27544
J_std                    0.01067                0.04300
gCO2e_per_query          0.00027                0.00764
avg_power_W             54.94273               68.61495
peak_mem_GB              5.09627                5.09627
acc_ci95                 0.01640                0.02620

Mistral uses 28.6x more energy per query than DeBERTa


In [60]:
print("DeBERTa weights: ", round(model.get_memory_footprint() / 1024**3, 2), "GB")
print("Mistral (4-bit): ", round(mistral.get_memory_footprint() / 1024**3, 2), "GB")

DeBERTa weights:  0.69 GB
Mistral (4-bit):  3.74 GB


In [ ]:
# ===== TODO 1: per-sample CSV (1,000 examples, both models) =====
import os, time, torch, numpy as np, pandas as pd
OUT = "phoenix_outputs"; os.makedirs(OUT, exist_ok=True)

def deberta_full(p, h):
    x = tokenizer(p, h, return_tensors="pt", truncation=True)
    x = {k: v.to("cuda") for k, v in x.items()}
    with torch.no_grad():
        probs = torch.softmax(model(**x).logits[0].float(), -1)
    i = int(probs.argmax())
    return model.config.id2label[i].upper(), probs[i].item()

def mistral_full(p, h):
    pred, s = predict_mistral(p, h)
    probs = torch.softmax(torch.tensor([s["Yes"], s["No"], s["Maybe"]]), -1)  # softmax over the 3 answer tokens only
    return pred, probs.max().item()

rows = []
for name, fn in [("DeBERTa-v3-base", deberta_full), ("Mistral-7B (few-shot)", mistral_full)]:
    for i, ex in enumerate(test_subset):
        torch.cuda.synchronize(); t = time.perf_counter()
        pred, conf = fn(ex["premise"], ex["hypothesis"])
        torch.cuda.synchronize(); lat = 1000 * (time.perf_counter() - t)
        rows.append(dict(model=name, idx=i, premise=ex["premise"], hypothesis=ex["hypothesis"],
                         true_label=label_names[ex["label"]], predicted_label=pred,
                         confidence=round(conf, 4), latency_ms=round(lat, 2)))
per_sample = pd.DataFrame(rows)
per_sample.to_csv(f"{OUT}/per_sample_1000.csv", index=False)
print(per_sample.groupby("model").apply(lambda d: (d.true_label == d.predicted_label).mean()))  # expect ~0.924 / 0.768

In [ ]:
# ===== TODO 3: exact prompt + setup details =====
import json, platform, importlib.metadata as md, transformers
setup = {
  "dataset": "stanfordnlp/snli, test split, label != -1 (9,824 valid), shuffle(seed=42).select(range(1000))",
  "shuffle_seed": 42,
  "note_seeds": "no torch/numpy seed set; inference is deterministic (argmax / logit comparison, no sampling)",
  "models": {"SLM": "cross-encoder/nli-deberta-v3-base (fp32, label ids: %s)" % model.config.id2label,
             "LLM": "mistralai/Mistral-7B-Instruct-v0.2 (bitsandbytes 4-bit, compute dtype fp16)"},
  "label_mapping_mistral": answer_to_label,
  "answer_token_ids": answer_token_ids,
  "decoding": "no generation: argmax over next-token logits of the 3 answer tokens (Yes/No/Maybe)",
  "prompt_fewshot_6": SYSTEM_PROMPT,
  "chat_template_handling": "prompt (instructions + 6 examples) is placed inside a single user turn (v0.2 has no system role); "
                            "apply_chat_template(add_generation_prompt=True), then tokenized with add_special_tokens=False to avoid double BOS",
  "example_formatted_prompt": mistral_tokenizer.apply_chat_template(
      [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\nPremise: {test_subset[0]['premise']}\nHypothesis: {test_subset[0]['hypothesis']}\nAnswer:"}],
      tokenize=False, add_generation_prompt=True),
  "hardware": torch.cuda.get_device_name(0),
  "energy_measurement": "pynvml nvmlDeviceGetTotalEnergyConsumption, 5 runs x 1000 examples per model, after 20-example warm-up",
  "python": platform.python_version(),
  "packages": {p: md.version(p) for p in ["torch","transformers","datasets","accelerate","bitsandbytes","sentencepiece","pandas","numpy","scikit-learn","nvidia-ml-py","tqdm"]},
}
json.dump(setup, open(f"{OUT}/setup_details.json", "w"), indent=2)
print(json.dumps(setup["packages"], indent=2))

In [ ]:
# ===== TODO 2: save ALL raw energy files before the runtime resets =====
import shutil
df.to_csv(f"{OUT}/nli_energy_runs_1000.csv", index=False)      # the 5 raw runs x 2 models
summary.to_csv(f"{OUT}/nli_summary_1000.csv")
for f in ["nli_energy_runs_500.csv"]:                           # earlier 500-sample file, if still present
    if os.path.exists(f): shutil.copy(f, OUT)
shutil.make_archive("phoenix_outputs", "zip", OUT)
from google.colab import files
files.download("phoenix_outputs.zip")   # also consider: drive.mount('/content/drive') and copy the folder there

In [ ]:
# ===== TODO 4: one clean final table =====
d, m = "DeBERTa-v3-base", "Mistral-7B (few-shot)"
final = pd.DataFrame({
  "Model": [d, m],
  "Accuracy (%)": [100*summary.loc[d,"accuracy"], 100*summary.loc[m,"accuracy"]],
  "Latency/query (ms)": [summary.loc[d,"latency_ms"], summary.loc[m,"latency_ms"]],
  "Energy/query (J)": [summary.loc[d,"J_per_query"], summary.loc[m,"J_per_query"]],
  "Avg GPU power (W)": [summary.loc[d,"avg_power_W"], summary.loc[m,"avg_power_W"]],
  "Model size (GB, weights)": [model.get_memory_footprint()/1024**3, mistral.get_memory_footprint()/1024**3],
}).round(2)
final.to_csv(f"{OUT}/final_table.csv", index=False)
print(final.to_string(index=False))

In [ ]:
# ===== TODO 5: error analysis (Mistral confusion matrix on the 1,000 run) =====
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
ps = per_sample[per_sample.model == "Mistral-7B (few-shot)"]
order = ["ENTAILMENT", "NEUTRAL", "CONTRADICTION"]
cm = confusion_matrix(ps.true_label, ps.predicted_label, labels=order)
cmn = cm / cm.sum(1, keepdims=True)
print(classification_report(ps.true_label, ps.predicted_label, labels=order))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cm[i,j]}\n({cmn[i,j]:.0%})", ha="center", va="center",
                color="white" if cmn[i,j] > .5 else "black")
ax.set_xticks(range(3)); ax.set_xticklabels(order, rotation=20); ax.set_yticks(range(3)); ax.set_yticklabels(order)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Mistral-7B few-shot: confusion matrix (n=1000)")
plt.tight_layout(); plt.savefig(f"{OUT}/mistral_confusion_matrix.png", dpi=200); plt.show()

In [ ]:
# ===== TODO 6 + 7: CO2 caveat and conclusion =====
acc_d, acc_m = 100*summary.loc[d,"accuracy"], 100*summary.loc[m,"accuracy"]
e_ratio = summary.loc[m,"J_per_query"] / summary.loc[d,"J_per_query"]
l_ratio = summary.loc[m,"latency_ms"] / summary.loc[d,"latency_ms"]
print("CO2 (if shown at all, label it): based on an ASSUMED 0.7 kg/kWh grid factor, not a cited value.\n")
print(f"On SNLI (n=1000 random test examples), DeBERTa-v3-base reached {acc_d:.1f}% accuracy vs {acc_m:.1f}% for 4-bit Mistral-7B (6-shot), "
      f"using about {e_ratio:.0f}x less GPU energy per query and {l_ratio:.0f}x lower latency on a Tesla T4. "
      "This supports the conclusion that a task-specialized SLM can outperform a larger general-purpose model on structured NLI tasks.")